# 02 — Sales forecasting

We compare three baselines on daily revenue:

1. **Weekday moving average** — fast & robust on weekly seasonality.
2. **Exponential smoothing (Holt-Winters)** — captures trend + seasonality.
3. **Prophet** — flexible additive model from Meta.

We hold out the last 14 days for evaluation.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

from db import load_sales

sales = load_sales()
daily = (sales.assign(day=sales['occurred_at'].dt.tz_localize(None).dt.normalize())
              .groupby('day')['total'].sum()
              .asfreq('D', fill_value=0)
              .rename('revenue'))
print(f'{len(daily)} days, {daily.index.min():%Y-%m-%d} → {daily.index.max():%Y-%m-%d}')
daily.tail()

In [ ]:
HORIZON = 14
train, test = daily.iloc[:-HORIZON], daily.iloc[-HORIZON:]
print('Train:', len(train), 'Test:', len(test))

### Baseline 1 — Weekday moving average

In [ ]:
wd_means = train.groupby(train.index.weekday).mean()
pred_wd = pd.Series(test.index.weekday.map(wd_means).values, index=test.index, name='weekday_avg')
pred_wd.head(7)

### Baseline 2 — Holt-Winters with weekly seasonality

In [ ]:
hw = ExponentialSmoothing(train, seasonal='add', seasonal_periods=7, trend='add')
hw_fit = hw.fit(optimized=True)
pred_hw = hw_fit.forecast(HORIZON).rename('holt_winters')
pred_hw.head(7)

### Baseline 3 — Prophet

In [ ]:
from prophet import Prophet

df_train = train.reset_index().rename(columns={'day': 'ds', 'revenue': 'y'})
m = Prophet(weekly_seasonality=True, daily_seasonality=False, yearly_seasonality=False)
m.fit(df_train)

future = m.make_future_dataframe(periods=HORIZON, freq='D')
forecast = m.predict(future)
pred_pr = forecast.set_index('ds')['yhat'].iloc[-HORIZON:].rename('prophet')
pred_pr.head(7)

### Compare on the holdout

In [ ]:
def score(true, pred):
    return {
        'MAE': mean_absolute_error(true, pred),
        'MAPE': mean_absolute_percentage_error(true.replace(0, np.nan).dropna(),
                                                pred.loc[true.replace(0, np.nan).dropna().index]) * 100,
    }

scores = pd.DataFrame({
    'weekday_avg': score(test, pred_wd),
    'holt_winters': score(test, pred_hw),
    'prophet': score(test, pred_pr),
}).T
scores

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index, y=train.values, name='Train', line=dict(color='#94a3b8')))
fig.add_trace(go.Scatter(x=test.index, y=test.values, name='Actual', line=dict(color='#0f172a', width=3)))
fig.add_trace(go.Scatter(x=pred_wd.index, y=pred_wd.values, name='Weekday avg', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=pred_hw.index, y=pred_hw.values, name='Holt-Winters', line=dict(dash='dot')))
fig.add_trace(go.Scatter(x=pred_pr.index, y=pred_pr.values, name='Prophet', line=dict(color='#ea580c')))
fig.update_layout(title='Daily revenue — train / actual / forecasts',
                  yaxis_title='Revenue (RWF)', height=500)
fig.show()

### Forecast next 14 days with the winner

Refit the best model on the **full** history, then project forward.

In [ ]:
best = scores['MAPE'].idxmin()
print(f'Best holdout MAPE: {best}')

if best == 'prophet':
    df_all = daily.reset_index().rename(columns={'day': 'ds', 'revenue': 'y'})
    m = Prophet(weekly_seasonality=True, daily_seasonality=False, yearly_seasonality=False)
    m.fit(df_all)
    future = m.make_future_dataframe(periods=14, freq='D')
    forecast = m.predict(future)
    plot_df = forecast.set_index('ds')[['yhat','yhat_lower','yhat_upper']].tail(45)
elif best == 'holt_winters':
    hw = ExponentialSmoothing(daily, seasonal='add', seasonal_periods=7, trend='add').fit(optimized=True)
    yhat = hw.forecast(14)
    plot_df = pd.concat([daily.tail(31).rename('yhat'), yhat.rename('yhat')])
    plot_df = plot_df.to_frame()
else:  # weekday
    wd_means = daily.groupby(daily.index.weekday).mean()
    horizon_idx = pd.date_range(daily.index.max() + pd.Timedelta(days=1), periods=14, freq='D')
    yhat = pd.Series(horizon_idx.weekday.map(wd_means).values, index=horizon_idx, name='yhat')
    plot_df = pd.concat([daily.tail(31).rename('yhat'), yhat]).to_frame()

plot_df.tail(20)